# Valuation serving readiness

Issue #35. Inspect the accepted fixed Poisson procedure on the selected eight-ZIP, county-verified cohort before exporting an artifact. This is a historical-sale estimate, not a current market valuation. The 2025 test is descriptive only and does not select a model or threshold. No prediction interval is approved while #20 remains open. Saved outputs are aggregates only.

In [1]:
import json
from pathlib import Path

from homelens.data.inventory import _file_identity, _read_csv
from homelens.modeling.baseline import NUMERIC_COLUMNS, _read_audit, _validate_cohort

root = Path("data/processed")
cohort_path = root / "modeling_cohort_county_verified.csv"
audit_path = root / "modeling_cohort_county_verified_audit.json"
comparison_path = root / "county_cohort_comparison.json"
catalog_audit_path = root / "property_catalog_audit.json"
validation_start, test_start, audit = _read_audit(audit_path)
cohort = _validate_cohort(
    _read_csv(cohort_path, ("sale_date", "property_type", "zip", "split")),
    validation_start,
    test_start,
)
comparison = json.loads(comparison_path.read_text(encoding="utf-8"))
catalog_audit = json.loads(catalog_audit_path.read_text(encoding="utf-8"))
assert audit["rules"]["county_boundary_validated"] is True
assert audit["prepared_rows"] == len(cohort)
assert audit["split_counts"] == cohort["split"].value_counts().to_dict()
assert comparison["selected_cohort_identity"] == _file_identity(cohort_path)
assert comparison["selected_audit_identity"] == _file_identity(audit_path)
assert audit["source_identity"] == catalog_audit["source_identity"]
assert audit["boundary_source_identity"] == catalog_audit["boundary_source_identity"]
assert comparison["fixed_loss"] == "poisson"
print(
    json.dumps(
        {
            "cohort_rows": len(cohort),
            "split_counts": audit["split_counts"],
            "source_sha256": audit["source_identity"]["sha256"],
            "boundary_sha256": audit["boundary_source_identity"]["sha256"],
            "selected_loss": comparison["fixed_loss"],
            "catalog_source_matches": True,
        },
        indent=2,
    )
)

{
  "cohort_rows": 17040,
  "split_counts": {
    "train": 13102,
    "validation": 2928,
    "test": 1010
  },
  "source_sha256": "01d4cb47bd23346f15dce22b32485cd2c2318fe11d70331152dfccdd6ec5aa96",
  "boundary_sha256": "96130b661255edbbe0229209eed8f4fb0661035737c0c7bae34eaedfa06ca789",
  "selected_loss": "poisson",
  "catalog_source_matches": true
}


In [2]:
train = cohort.loc[cohort["split"].eq("train")]
ranges = {
    column: [float(train[column].min()), float(train[column].max())]
    for column in NUMERIC_COLUMNS
}
envelope = cohort["property_type"].isin(train["property_type"].unique()) & cohort[
    "zip"
].isin(train["zip"].unique())
for column, (low, high) in ranges.items():
    envelope &= cohort[column].between(low, high)
print(
    json.dumps(
        {
            "training_feature_ranges": ranges,
            "training_zips": sorted(train["zip"].unique().tolist()),
            "training_types": sorted(train["property_type"].unique().tolist()),
            "outside_train_envelope": {
                name: int((~envelope & cohort["split"].eq(name)).sum())
                for name in ("validation", "test")
            },
        },
        indent=2,
    )
)

{
  "training_feature_ranges": {
    "beds": [
      0.0,
      8.0
    ],
    "baths": [
      1.0,
      10.5
    ],
    "square_feet": [
      383.0,
      10129.0
    ],
    "year_built": [
      1890.0,
      2023.0
    ]
  },
  "training_zips": [
    "27503",
    "27701",
    "27703",
    "27704",
    "27705",
    "27707",
    "27712",
    "27713"
  ],
  "training_types": [
    "Condo/Co-op",
    "Single Family Residential",
    "Townhouse"
  ],
  "outside_train_envelope": {
    "validation": 484,
    "test": 205
  }
}


In [3]:
selected_validation = comparison["validation"]["selected_model_on_common_sales"][
    "overall"
]
held_out = comparison["held_out_test"]["selected_model"]["overall"]
print(
    json.dumps(
        {
            "validation_overall": selected_validation,
            "held_out_test_overall_descriptive": held_out,
            "decision": (
                "Export the fixed train-only model; refuse unsupported inputs. "
                "Report historical scope and no intervals."
            ),
        },
        indent=2,
    )
)

{
  "validation_overall": {
    "rows": 2928,
    "small_slice": false,
    "mean_signed_error_usd": -48895.31,
    "mae_usd": 74861.81,
    "rmse_usd": 139102.42,
    "median_absolute_error_usd": 48030.07,
    "p90_absolute_error_usd": 150892.13
  },
  "held_out_test_overall_descriptive": {
    "rows": 1010,
    "small_slice": false,
    "mean_signed_error_usd": -56951.48,
    "mae_usd": 81625.74,
    "rmse_usd": 206744.71,
    "median_absolute_error_usd": 45240.73,
    "p90_absolute_error_usd": 166403.56
  },
  "decision": "Export the fixed train-only model; refuse unsupported inputs. Report historical scope and no intervals."
}


The model was selected with 2024 validation and the geography decision also used 2024 validation, so those results are exploratory. The 2025 test was viewed only after the procedure was fixed. Export must verify source/boundary hashes against the property catalog, preserve the train-only fit, and refuse mismatched artifacts or newer/out-of-scope sale records. No live drift guarantee or current-price claim follows from these historical splits.